# Fase 4d — Verificación, evaluación manual y submuestreo del dataset de Test

Una vez generado el dataset de test (Fase 4c), las preguntas se revisaron manualmente en un Excel (`evaluacion_tripletas.xlsx`), asignando a cada una una **nota (1-10)** y un **veredicto** (`VÁLIDA`, `RETOQUE`, `DESCARTADA`). Este notebook cruza esa evaluación con el JSON y construye la submuestra final que se usará en la evaluación del sistema RAG completo.

1. **Verificación de consistencia JSON ↔ Excel** (celda 1): cruza cada pregunta del JSON con su fila del Excel (por pregunta, cita o respuesta normalizadas) y detecta inconsistencias: preguntas reformuladas en el JSON que no se actualizaron en el Excel, preguntas descartadas en el Excel que siguen en el JSON, veredictos no reconocidos o preguntas válidas sin nota asignada. Guarda `preguntas_con_notas.json` con la nota y el veredicto incorporados a cada pregunta.
2. **Análisis descriptivo global** (celda 2): distribución del dataset completo por tipo de pregunta (Directa/Caso/Compleja/Negativa) y por documento de origen.
3. **Submuestreo estratificado** (celda 3): reduce el dataset a un tamaño manejable y equilibrado para la evaluación final —**40 preguntas por tipo (160 en total)**— garantizando representación mínima de cada (documento, tipo) cuando hay candidatas, priorizando dentro de cada grupo por nota descendente, repartiendo las plazas restantes de forma proporcional entre documentos mediante el método del resto mayor (Hamilton), y seleccionando las preguntas "Negativa" (sin nota) de forma aleatoria respetando el reparto proporcional por documento. Genera `dataset_submuestreado.json` y un Excel comparativo con la distribución original frente a la submuestra.

In [ ]:
"""
Verificación de consistencia entre el JSON final y el Excel de evaluación.

Detecta:
  1. Preguntas del JSON que NO aparecen en el Excel (posibles reformulaciones
     que no se actualizaron en el Excel).
  2. Preguntas del Excel marcadas como DESCARTADA que siguen en el JSON.
  3. Preguntas del Excel sin veredicto asignado.
  4. Preguntas del JSON a las que no se puede asignar una nota unívoca.
"""

import json
import pandas as pd
from openpyxl import load_workbook

# ── CONFIGURACIÓN ─────────────────────────────────────────────────
ARCHIVO_JSON = 'preguntas_evaluacion_test_final_v3.json'
ARCHIVO_EXCEL = 'evaluacion_tripletas.xlsx'
ARCHIVO_SALIDA_ENRIQUECIDO = 'preguntas_con_notas.json'

COL_PREGUNTA = 'Pregunta'
COL_CITA = 'Cita Literal'
COL_RESPUESTA = 'Respuesta Ideal'
COL_NOTA = 'Nota (1-10)'
COL_VEREDICTO = 'Veredicto'

VEREDICTO_VALIDA = '✅ VÁLIDA'
VEREDICTO_RETOQUE = '⚠️ RETOQUE'
VEREDICTO_DESCARTADA = '❌ DESCARTADA'
# ─────────────────────────────────────────────────────────────────


def norm(texto):
    """
    Normaliza texto para comparación fiable entre JSON y Excel:
    - Pasa a minúsculas
    - Colapsa espacios múltiples
    - Elimina barras invertidas de escape (\" → ")
    - Unifica comillas tipográficas curvas (" " ' ') a rectas (" ')
    - Unifica guiones largos (– —) a guión normal (-)
    """
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return ''
    s = str(texto)
    # Quitar escapes
    s = s.replace('\\"', '"').replace("\\'", "'")
    # Unificar comillas tipográficas
    s = s.replace('“', '"').replace('”', '"')
    s = s.replace('‘', "'").replace('’', "'")
    # Unificar guiones
    s = s.replace('—', '-').replace('–', '-')
    # Colapsar espacios y pasar a minúsculas
    return ' '.join(s.strip().lower().split())


def cargar_excel():
    """Carga la primera hoja del Excel detectando la fila de cabeceras."""
    wb = load_workbook(ARCHIVO_EXCEL, data_only=True)
    sheet_name = wb.sheetnames[0]
    print(f"Hoja detectada: '{sheet_name}'")

    # Detectar la fila que contiene 'Pregunta' como cabecera real
    df_raw = pd.read_excel(ARCHIVO_EXCEL, sheet_name=sheet_name, header=None)
    fila_cabecera = None
    for i, row in df_raw.iterrows():
        valores = [str(v).strip() if pd.notna(v) else '' for v in row]
        if COL_PREGUNTA in valores and COL_CITA in valores:
            fila_cabecera = i
            break

    if fila_cabecera is None:
        print(f"[ERROR] No se detectó la fila de cabeceras con '{COL_PREGUNTA}' y '{COL_CITA}'")
        return None

    print(f"Fila de cabeceras detectada: {fila_cabecera + 1}")
    df = pd.read_excel(ARCHIVO_EXCEL, sheet_name=sheet_name, header=fila_cabecera)
    df.columns = [str(c).strip() for c in df.columns]
    # Eliminar filas totalmente vacías
    df = df.dropna(how='all').reset_index(drop=True)
    return df


def main():
    """
    Carga el JSON final de preguntas y el Excel de evaluación manual, cruza
    ambos por pregunta/cita/respuesta normalizadas, reporta inconsistencias
    (preguntas sin cruzar, descartadas que siguen en el JSON, sin veredicto
    o sin nota) y guarda un JSON enriquecido con la nota y el veredicto de
    cada pregunta.
    """
    # ── Cargar archivos ───────────────────────────────────────────
    with open(ARCHIVO_JSON, 'r', encoding='utf-8') as f:
        preguntas_json = json.load(f)
    print(f"JSON: {len(preguntas_json)} preguntas")

    df_excel = cargar_excel()
    if df_excel is None:
        return
    print(f"Excel: {len(df_excel)} filas\n")

    # Verificar columnas esperadas
    for col in [COL_PREGUNTA, COL_CITA, COL_RESPUESTA, COL_NOTA, COL_VEREDICTO]:
        if col not in df_excel.columns:
            print(f"[ERROR] Columna '{col}' no encontrada en el Excel.")
            print(f"        Columnas disponibles: {list(df_excel.columns)}")
            return

    # ── Construir índices de búsqueda ─────────────────────────────
    # Para cada fila del Excel, guardamos pregunta/cita/respuesta normalizadas
    excel_registros = []
    for idx, row in df_excel.iterrows():
        excel_registros.append({
            'fila_excel': idx + 2,  # fila real en Excel (cabecera = 1)
            'pregunta': norm(row[COL_PREGUNTA]),
            'cita': norm(row[COL_CITA]),
            'respuesta': norm(row[COL_RESPUESTA]),
            'nota': row[COL_NOTA],
            'veredicto': str(row[COL_VEREDICTO]).strip() if pd.notna(row[COL_VEREDICTO]) else '',
        })

    # ── Cruce JSON → Excel ────────────────────────────────────────
    print("="*70)
    print(" CRUCE JSON → EXCEL")
    print("="*70)
    print("Nota: las preguntas Negativas no se evaluaron y se excluyen del cruce.\n")

    no_encontradas = []      # JSON no-Negativas que no aparecen en Excel
    match_por_pregunta = 0
    match_por_cita = 0
    match_por_respuesta = 0
    match_multiple = []
    negativas_count = 0

    preguntas_enriquecidas = []

    for i, p in enumerate(preguntas_json):
        # Las Negativas se conservan tal cual, sin cruce
        if p.get('tipo') == 'Negativa':
            negativas_count += 1
            preguntas_enriquecidas.append({
                **p,
                'nota': None,
                'veredicto': 'N/A (Negativa)',
                'fila_excel': None,
            })
            continue

        pregunta_n = norm(p['pregunta'])
        cita_n = norm(p['cita_literal'])
        respuesta_n = norm(p['respuesta_ideal'])

        # Buscar por pregunta primero
        matches = [r for r in excel_registros if r['pregunta'] == pregunta_n and pregunta_n != '']
        tipo_match = 'pregunta'

        if not matches:
            matches = [r for r in excel_registros if r['cita'] == cita_n and cita_n != '']
            tipo_match = 'cita'

        if not matches:
            matches = [r for r in excel_registros if r['respuesta'] == respuesta_n and respuesta_n != '']
            tipo_match = 'respuesta'

        if not matches:
            no_encontradas.append((i, p))
            preguntas_enriquecidas.append({**p, 'nota': None, 'veredicto': None, 'fila_excel': None})
            continue

        if len(matches) > 1:
            match_multiple.append((i, p, matches))
            match = matches[0]
        else:
            match = matches[0]

        if tipo_match == 'pregunta':
            match_por_pregunta += 1
        elif tipo_match == 'cita':
            match_por_cita += 1
        else:
            match_por_respuesta += 1

        preguntas_enriquecidas.append({
            **p,
            'nota': match['nota'],
            'veredicto': match['veredicto'],
            'fila_excel': match['fila_excel'],
        })

    print(f"  Negativas (excluidas del cruce): {negativas_count}")
    print(f"  Match por pregunta literal:      {match_por_pregunta}")
    print(f"  Match por cita literal:          {match_por_cita}")
    print(f"  Match por respuesta ideal:       {match_por_respuesta}")
    print(f"  No encontradas en Excel:         {len(no_encontradas)}")
    print(f"  Matches múltiples:               {len(match_multiple)}")

    # ── Reportar preguntas JSON no encontradas en Excel ───────────
    if no_encontradas:
        print("\n" + "="*70)
        print(" [!] PREGUNTAS DEL JSON NO ENCONTRADAS EN EL EXCEL")
        print("="*70)
        print("Probablemente se reformularon en el JSON pero no en el Excel.")
        print("Revisa y actualiza el Excel manualmente antes del submuestreo.\n")
        for i, p in no_encontradas[:20]:  # primeras 20 para no saturar
            print(f"  · JSON índice {i} | tipo={p['tipo']} | doc={p['source'][:50]}")
            print(f"    Pregunta: {p['pregunta'][:100]}...")
        if len(no_encontradas) > 20:
            print(f"  ... y {len(no_encontradas) - 20} más")

    # ── Reportar matches múltiples ────────────────────────────────
    if match_multiple:
        print("\n" + "="*70)
        print(" [!] PREGUNTAS DEL JSON CON VARIOS MATCHES EN EL EXCEL")
        print("="*70)
        for i, p, matches in match_multiple[:10]:
            print(f"  · JSON índice {i}: {p['pregunta'][:80]}...")
            print(f"    Filas Excel: {[m['fila_excel'] for m in matches]}")

    # ── Preguntas del JSON marcadas como DESCARTADA ──────────────
    descartadas_en_json = [
        p for p in preguntas_enriquecidas
        if p.get('veredicto') == VEREDICTO_DESCARTADA
    ]
    if descartadas_en_json:
        print("\n" + "="*70)
        print(" [!] PREGUNTAS DEL JSON MARCADAS COMO ❌ DESCARTADA EN EXCEL")
        print("="*70)
        print("Estas preguntas aparecen en el JSON pero están descartadas en el Excel.")
        print("Deberías eliminarlas del JSON.\n")
        for p in descartadas_en_json:
            print(f"  · Fila Excel {p['fila_excel']} | tipo={p['tipo']}")
            print(f"    Pregunta: {p['pregunta'][:100]}...")

    # ── Preguntas sin veredicto ──────────────────────────────────
    sin_veredicto = [
        p for p in preguntas_enriquecidas
        if p.get('veredicto') is not None
        and p.get('veredicto') not in (VEREDICTO_VALIDA, VEREDICTO_RETOQUE, VEREDICTO_DESCARTADA, 'N/A (Negativa)')
    ]
    if sin_veredicto:
        print("\n" + "="*70)
        print(" [!] PREGUNTAS SIN VEREDICTO VÁLIDO")
        print("="*70)
        for p in sin_veredicto[:10]:
            print(f"  · Fila Excel {p['fila_excel']} | veredicto actual: '{p['veredicto']}'")

    # ── Preguntas sin nota ───────────────────────────────────────
    sin_nota = [
        p for p in preguntas_enriquecidas
        if p.get('nota') is None or (isinstance(p.get('nota'), float) and pd.isna(p.get('nota')))
    ]
    sin_nota_con_veredicto_valido = [
        p for p in sin_nota
        if p.get('veredicto') in (VEREDICTO_VALIDA, VEREDICTO_RETOQUE)
    ]
    if sin_nota_con_veredicto_valido:
        print("\n" + "="*70)
        print(" [!] PREGUNTAS VÁLIDAS SIN NOTA ASIGNADA")
        print("="*70)
        for p in sin_nota_con_veredicto_valido[:10]:
            print(f"  · Fila Excel {p.get('fila_excel')} | tipo={p['tipo']}")

    # ── Resumen final ─────────────────────────────────────────────
    print("\n" + "="*70)
    print(" RESUMEN FINAL")
    print("="*70)
    n_ok = sum(1 for p in preguntas_enriquecidas
               if p.get('veredicto') in (VEREDICTO_VALIDA, VEREDICTO_RETOQUE)
               and p.get('nota') is not None
               and not (isinstance(p.get('nota'), float) and pd.isna(p.get('nota'))))
    n_evaluables = len(preguntas_json) - negativas_count
    print(f"  Total en JSON:                     {len(preguntas_json)}")
    print(f"  Negativas (no evaluadas):          {negativas_count}")
    print(f"  Evaluables (no-Negativas):         {n_evaluables}")
    print(f"  Cruzadas con Excel:                {n_evaluables - len(no_encontradas)}")
    print(f"  No encontradas:                    {len(no_encontradas)}")
    print(f"  Marcadas como DESCARTADA:          {len(descartadas_en_json)}")
    print(f"  Válidas con nota asignada:         {n_ok}")

    if len(no_encontradas) == 0 and len(descartadas_en_json) == 0 and len(sin_nota_con_veredicto_valido) == 0:
        print("\n  [OK] Todo limpio, listo para submuestreo por nota")
    else:
        print("\n  [!] Hay inconsistencias, revisa antes de continuar")

    # ── Guardar JSON enriquecido con notas ────────────────────────
    with open(ARCHIVO_SALIDA_ENRIQUECIDO, 'w', encoding='utf-8') as f:
        json.dump(preguntas_enriquecidas, f, ensure_ascii=False, indent=4)
    print(f"\nJSON enriquecido con notas guardado en: '{ARCHIVO_SALIDA_ENRIQUECIDO}'")
    print("="*70)


if __name__ == '__main__':
    main()

JSON: 263 preguntas
Hoja detectada: 'Evaluación'
Fila de cabeceras detectada: 2
Excel: 305 filas

 CRUCE JSON → EXCEL
Nota: las preguntas Negativas no se evaluaron y se excluyen del cruce.

  Negativas (excluidas del cruce): 78
  Match por pregunta literal:      181
  Match por cita literal:          4
  Match por respuesta ideal:       0
  No encontradas en Excel:         0
  Matches múltiples:               0

 RESUMEN FINAL
  Total en JSON:                     263
  Negativas (no evaluadas):          78
  Evaluables (no-Negativas):         185
  Cruzadas con Excel:                185
  No encontradas:                    0
  Marcadas como DESCARTADA:          0
  Válidas con nota asignada:         185

  [OK] Todo limpio, listo para submuestreo por nota

JSON enriquecido con notas guardado en: 'preguntas_con_notas.json'


In [2]:
import json
import pandas as pd
from collections import defaultdict

ARCHIVO_JSON = 'preguntas_evaluacion_test_final_v3.json'

def doc_corto(source):
    """
    Acorta el nombre del documento (quita la extensión .pdf y trunca a 60
    caracteres) para mostrarlo en las tablas de resumen.
    """
    return source.replace('.pdf', '')[:60]

def main():
    """
    Carga el dataset de test final y muestra un resumen global
    (distribución por tipo de pregunta) y un desglose por documento y tipo.
    """
    with open(ARCHIVO_JSON, 'r', encoding='utf-8') as f:
        preguntas = json.load(f)

    df = pd.DataFrame(preguntas)
    df['doc_corto'] = df['source'].apply(doc_corto)

    tipos = ['Directa', 'Caso', 'Compleja', 'Negativa']
    total = len(df)

    # ── RESUMEN GLOBAL ────────────────────────────────────────────
    print("\n" + "="*65)
    print(" ANÁLISIS GLOBAL DEL DATASET")
    print("="*65)
    print(f" Total de preguntas: {total}\n")

    resumen = df.groupby('tipo').size().reindex(tipos, fill_value=0)
    for tipo in tipos:
        n = resumen[tipo]
        pct = n / total * 100
        print(f"  {tipo:<12} {n:>4}  ({pct:.1f}%)")
    print(f"  {'TOTAL':<12} {total:>4}  (100%)")

    # ── POR DOCUMENTO ─────────────────────────────────────────────
    print("\n" + "="*65)
    print(" DISTRIBUCIÓN POR DOCUMENTO")
    print("="*65)

    pivot = df.groupby(['doc_corto', 'tipo']).size().unstack(fill_value=0)
    for t in tipos:
        if t not in pivot.columns:
            pivot[t] = 0
    pivot = pivot[tipos]
    pivot['TOTAL'] = pivot.sum(axis=1)
    pivot['% dataset'] = (pivot['TOTAL'] / total * 100).round(1)
    pivot = pivot.sort_values('TOTAL', ascending=False)

    header = f"{'Documento':<62} {'Dir':>4} {'Cas':>4} {'Com':>4} {'Neg':>4} {'TOT':>5} {'%':>6}"
    print(header)
    print("-" * 95)
    for doc, row in pivot.iterrows():
        print(f"{doc:<62} {int(row['Directa']):>4} {int(row['Caso']):>4} {int(row['Compleja']):>4} {int(row['Negativa']):>4} {int(row['TOTAL']):>5} {row['% dataset']:>5.1f}%")
    print("-" * 95)
    print(f"{'TOTAL':<62} {int(pivot['Directa'].sum()):>4} {int(pivot['Caso'].sum()):>4} {int(pivot['Compleja'].sum()):>4} {int(pivot['Negativa'].sum()):>4} {total:>5} {'100%':>6}")
    print("="*65)

if __name__ == '__main__':
    main()


 ANÁLISIS GLOBAL DEL DATASET
 Total de preguntas: 263

  Directa        71  (27.0%)
  Caso           64  (24.3%)
  Compleja       50  (19.0%)
  Negativa       78  (29.7%)
  TOTAL         263  (100%)

 DISTRIBUCIÓN POR DOCUMENTO
Documento                                                       Dir  Cas  Com  Neg   TOT      %
-----------------------------------------------------------------------------------------------
1181_Guía hospitalaria de terapéutica antibiótica en adultos     11    8    7   13    39  14.8%
16264_Vacunación estacional frente a infecciones respiratori      7    7    4    7    25   9.5%
15265_Protocolo Regional para la indicación, uso y autorizac      7    7    4    7    25   9.5%
15264_Protocolo Regional para la indicación, uso y autorizac      7    5    5    7    24   9.1%
11064_Protocolo de vigilancia, prevención y control de micro      5    5    3    6    19   7.2%
10584_Vacunación estacional frente a infecciones respiratori      4    5    3    5    17   6.5%
452

In [ ]:
"""
Submuestreo estratificado del dataset RAG con las siguientes propiedades:

1. Tamaño fijo de 40 preguntas por tipo (160 total, 25% equitativo).
2. Representación mínima garantizada: 1 pregunta por (documento, tipo)
   siempre que haya candidatas disponibles.
3. Priorización por nota descendente dentro de cada (documento, tipo).
4. Desempate aleatorio con semilla fija (reproducible).
5. Las Negativas no tienen nota, se seleccionan aleatoriamente respetando
   el reparto proporcional por documento.

Input:  preguntas_con_notas.json (generado por verificar_dataset.py)
Output: dataset_submuestreado.json + analisis_submuestreo.xlsx
"""

import json
import pandas as pd
import random
import math
from collections import defaultdict

# ── CONFIGURACIÓN ─────────────────────────────────────────────────
ARCHIVO_ENTRADA = 'preguntas_con_notas.json'
ARCHIVO_SALIDA_JSON = 'dataset_submuestreado.json'
ARCHIVO_SALIDA_EXCEL = 'analisis_submuestreo.xlsx'

SEED = 42
TIPOS = ['Directa', 'Caso', 'Compleja', 'Negativa']
N_POR_TIPO = 40
MIN_POR_DOC_POR_TIPO = 1

VEREDICTOS_VALIDOS = {'✅ VÁLIDA', '⚠️ RETOQUE'}
# ─────────────────────────────────────────────────────────────────


def doc_corto(source):
    """
    Acorta el nombre del documento para las tablas de resumen (misma
    lógica que en la celda de análisis global).
    """
    return source.replace('.pdf', '')[:60]


def ordenar_candidatas(candidatas, es_negativa=False):
    """
    Ordena las candidatas para la selección:
    - Negativas: orden aleatorio (no tienen nota)
    - Resto: por nota descendente, empates aleatorios
    Devuelve lista de índices ordenados.
    """
    if es_negativa:
        indices = list(range(len(candidatas)))
        random.shuffle(indices)
        return indices

    # Construir lista de (nota, aleatorio_desempate, idx)
    items = []
    for idx, c in enumerate(candidatas):
        nota = c.get('nota')
        try:
            nota_num = float(nota) if nota is not None else -1
        except (ValueError, TypeError):
            nota_num = -1
        items.append((-nota_num, random.random(), idx))  # negativa para DESC

    items.sort()
    return [idx for _, _, idx in items]


def submuestrear_tipo(tipo, pool_por_doc, documentos, n_objetivo):
    """
    Selecciona n_objetivo preguntas del tipo dado en dos fases:
      Fase 1: mínimo por (documento, tipo)
      Fase 2: reparto proporcional al pool disponible, priorizando por nota
    """
    es_negativa = (tipo == 'Negativa')
    seleccionadas = []
    usados_por_doc = defaultdict(int)

    # ── Pre-ordenar el pool de cada documento por nota ────────────
    pools_ordenados = {}
    for doc in documentos:
        pool = pool_por_doc.get(doc, [])
        orden = ordenar_candidatas(pool, es_negativa=es_negativa)
        pools_ordenados[doc] = [pool[i] for i in orden]

    # ── Fase 1: representación mínima ─────────────────────────────
    for doc in documentos:
        pool = pools_ordenados[doc]
        n_tomar = min(MIN_POR_DOC_POR_TIPO, len(pool))
        if n_tomar > 0:
            seleccionadas.extend(pool[:n_tomar])
            usados_por_doc[doc] = n_tomar

    # ── Fase 2: reparto proporcional del resto ───────────────────
    restantes_pools = {
        doc: pools_ordenados[doc][usados_por_doc[doc]:]
        for doc in documentos
    }
    total_disponible = sum(len(v) for v in restantes_pools.values())
    plazas_restantes = n_objetivo - len(seleccionadas)

    if plazas_restantes <= 0:
        return seleccionadas[:n_objetivo]

    if plazas_restantes >= total_disponible:
        # No hay suficiente, tomamos todo
        for pool in restantes_pools.values():
            seleccionadas.extend(pool)
        return seleccionadas

    # Reparto proporcional con método de Hamilton (Largest Remainder Method):
    # asignamos primero la parte entera y después repartimos las plazas
    # sobrantes entre los documentos con mayor parte decimal, no entre los
    # de mayor capacidad absoluta (lo cual sesgaría el reparto hacia los
    # documentos grandes).
    asignacion = {}
    restos = {}
    for doc in documentos:
        disp = len(restantes_pools[doc])
        proporcion = disp / total_disponible if total_disponible > 0 else 0
        plaza_ideal = plazas_restantes * proporcion
        parte_entera = math.floor(plaza_ideal)
        asignacion[doc] = min(parte_entera, disp)
        # Guardamos la parte decimal para el desempate posterior
        restos[doc] = plaza_ideal - parte_entera

    # Distribuir las plazas que falten entre los documentos con mayor
    # parte decimal (método del resto mayor de Hamilton). Solo son
    # candidatos los que todavía tienen capacidad disponible.
    faltan = plazas_restantes - sum(asignacion.values())
    candidatos = sorted(
        [doc for doc in documentos
         if len(restantes_pools[doc]) - asignacion[doc] > 0],
        key=lambda d: -restos[d]
    )
    for i in range(min(faltan, len(candidatos))):
        asignacion[candidatos[i]] += 1

    # Añadir seleccionadas respetando orden por nota
    for doc, n in asignacion.items():
        if n > 0:
            seleccionadas.extend(restantes_pools[doc][:n])

    return seleccionadas


def main():
    """
    Carga el JSON enriquecido con notas, selecciona las preguntas
    candidatas (válidas/retoque + todas las Negativas), aplica el
    submuestreo estratificado por tipo (submuestrear_tipo) y guarda el
    resultado en JSON, junto con un informe de verificación y un Excel
    comparativo (distribución original vs submuestra, por documento y por
    nota).
    """
    random.seed(SEED)

    # ── Cargar JSON enriquecido ───────────────────────────────────
    with open(ARCHIVO_ENTRADA, 'r', encoding='utf-8') as f:
        preguntas = json.load(f)

    df = pd.DataFrame(preguntas)
    df['doc_corto'] = df['source'].apply(doc_corto)
    total = len(df)
    documentos = sorted(df['source'].unique())

    print(f"\n{'='*70}")
    print(f" SUBMUESTREO ESTRATIFICADO POR NOTA")
    print(f"{'='*70}")
    print(f"  Dataset original:    {total} preguntas")
    print(f"  Documentos:          {len(documentos)}")
    print(f"  Objetivo:            {N_POR_TIPO * len(TIPOS)} preguntas")
    print(f"  Seed:                {SEED}\n")

    # ── Filtrar solo candidatas válidas (y Negativas) ─────────────
    candidatas = []
    for p in preguntas:
        if p['tipo'] == 'Negativa':
            candidatas.append(p)
        elif p.get('veredicto') in VEREDICTOS_VALIDOS:
            candidatas.append(p)

    print(f"  Candidatas totales (válidas + Negativas): {len(candidatas)}")
    for tipo in TIPOS:
        n = sum(1 for p in candidatas if p['tipo'] == tipo)
        print(f"    {tipo:<10} {n}")
    print()

    # ── Agrupar por (tipo, documento) ─────────────────────────────
    grupos = defaultdict(lambda: defaultdict(list))
    for p in candidatas:
        grupos[p['tipo']][p['source']].append(p)

    # ── Submuestreo por tipo ──────────────────────────────────────
    submuestra = []
    for tipo in TIPOS:
        seleccionadas = submuestrear_tipo(tipo, grupos[tipo], documentos, N_POR_TIPO)
        submuestra.extend(seleccionadas)
        print(f"  {tipo:<10} → {len(seleccionadas):>3} seleccionadas")

    # ── Limpiar campos auxiliares antes de exportar ───────────────
    # El JSON final conserva solo los campos originales del dataset
    CAMPOS_A_ELIMINAR = {'fila_excel', 'nota', 'veredicto'}
    submuestra_clean = [
        {k: v for k, v in p.items() if k not in CAMPOS_A_ELIMINAR}
        for p in submuestra
    ]

    # ── Guardar JSON ──────────────────────────────────────────────
    with open(ARCHIVO_SALIDA_JSON, 'w', encoding='utf-8') as f:
        json.dump(submuestra_clean, f, ensure_ascii=False, indent=4)
    print(f"\n  JSON submuestreado: '{ARCHIVO_SALIDA_JSON}' ({len(submuestra_clean)} preguntas)")

    # ── Verificaciones ────────────────────────────────────────────
    df_sub = pd.DataFrame(submuestra)
    df_sub['doc_corto'] = df_sub['source'].apply(doc_corto)

    print("\n" + "="*70)
    print(" VERIFICACIÓN DEL SUBMUESTREO")
    print("="*70)

    print("\n  Distribución por tipo:")
    for tipo in TIPOS:
        n = len(df_sub[df_sub['tipo'] == tipo])
        pct = n / len(df_sub) * 100
        print(f"    {tipo:<10} {n:>3}  ({pct:.1f}%)")

    docs_orig = set(df['source'])
    docs_sub = set(df_sub['source'])
    print(f"\n  Cobertura documental: {len(docs_sub)}/{len(docs_orig)} documentos")
    faltan = docs_orig - docs_sub
    if faltan:
        print("  [!] Documentos sin representación en la submuestra:")
        for d in faltan:
            print(f"        - {doc_corto(d)}")
    else:
        print("  [OK] Todos los documentos representados")

    # Estadísticas de nota (solo no-Negativas)
    df_eval = df_sub[df_sub['tipo'] != 'Negativa'].copy()
    df_eval['nota_num'] = pd.to_numeric(df_eval['nota'], errors='coerce')
    print("\n  Estadísticas de nota en la submuestra (no-Negativas):")
    print(f"    Media:    {df_eval['nota_num'].mean():.2f}")
    print(f"    Mediana:  {df_eval['nota_num'].median():.2f}")
    print(f"    Mín:      {df_eval['nota_num'].min():.1f}")
    print(f"    Máx:      {df_eval['nota_num'].max():.1f}")

    # ── Comparativa por documento (Excel) ─────────────────────────
    pivot_orig = df.groupby(['doc_corto', 'tipo']).size().unstack(fill_value=0)
    pivot_sub = df_sub.groupby(['doc_corto', 'tipo']).size().unstack(fill_value=0)
    for t in TIPOS:
        if t not in pivot_orig.columns: pivot_orig[t] = 0
        if t not in pivot_sub.columns: pivot_sub[t] = 0
    pivot_orig = pivot_orig[TIPOS]
    pivot_sub = pivot_sub[TIPOS]
    pivot_orig['TOTAL'] = pivot_orig.sum(axis=1)
    pivot_sub['TOTAL'] = pivot_sub.sum(axis=1)

    # ── Exportar Excel ────────────────────────────────────────────
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    wb = openpyxl.Workbook()

    COLORES = {
        'header': 'CCCCCC',
        'Directa': 'D9EAD3',
        'Caso': 'FCE5CD',
        'Compleja': 'CFE2F3',
        'Negativa': 'F4CCCC',
        'total': 'EEEEEE',
        'titulo': '1F3864',
    }
    thin = Side(style='thin', color='BBBBBB')
    borde = Border(left=thin, right=thin, top=thin, bottom=thin)

    def estilo(cell, bold=False, bg=None, align='center', color='000000', size=10):
        """
        Aplica un estilo visual consistente (fuente, alineación, bordes y
        color de fondo) a una celda del Excel de análisis.
        """
        cell.font = Font(name='Arial', bold=bold, color=color, size=size)
        cell.alignment = Alignment(horizontal=align, vertical='center', wrap_text=True)
        cell.border = borde
        if bg:
            cell.fill = PatternFill('solid', start_color=bg)

    # Hoja 1: Resumen global
    ws = wb.active
    ws.title = 'Resumen'
    for col, w in zip('ABCDE', [14, 12, 12, 12, 14]):
        ws.column_dimensions[col].width = w

    ws.merge_cells('A1:E1')
    estilo(ws['A1'], bold=True, bg=COLORES['titulo'], color='FFFFFF')
    ws['A1'] = 'SUBMUESTREO ESTRATIFICADO POR NOTA'
    ws.row_dimensions[1].height = 24

    for col, cab in enumerate(['Tipo', 'Original', '% orig.', 'Submuestra', '% sub.'], 1):
        estilo(ws.cell(row=2, column=col, value=cab), bold=True, bg=COLORES['header'])

    for i, tipo in enumerate(TIPOS, 3):
        n_orig = len(df[df['tipo'] == tipo])
        n_sub = len(df_sub[df_sub['tipo'] == tipo])
        for col, val in enumerate([
            tipo, n_orig, f"{n_orig/total*100:.1f}%",
            n_sub, f"{n_sub/len(df_sub)*100:.1f}%"
        ], 1):
            estilo(ws.cell(row=i, column=col, value=val), bg=COLORES.get(tipo, 'FFFFFF'))

    fila_t = len(TIPOS) + 3
    for col, val in enumerate(['TOTAL', total, '100%', len(df_sub), '100%'], 1):
        estilo(ws.cell(row=fila_t, column=col, value=val), bold=True, bg=COLORES['total'])

    # Hoja 2: Comparativa por documento
    ws2 = wb.create_sheet('Por Documento')
    cabeceras = ['Documento',
                 'Dir orig', 'Cas orig', 'Com orig', 'Neg orig', 'TOT orig',
                 'Dir sub', 'Cas sub', 'Com sub', 'Neg sub', 'TOT sub']
    anchos = [55, 8, 8, 8, 8, 9, 8, 8, 8, 8, 9]
    for col, (cab, w) in enumerate(zip(cabeceras, anchos), 1):
        ws2.column_dimensions[get_column_letter(col)].width = w
        estilo(ws2.cell(row=1, column=col, value=cab), bold=True, bg=COLORES['header'])

    merged = pivot_orig.join(pivot_sub, lsuffix='_orig', rsuffix='_sub', how='outer').fillna(0).astype(int)
    merged = merged.sort_values('TOTAL_orig', ascending=False)

    for i, (doc, row) in enumerate(merged.iterrows(), 2):
        c = ws2.cell(row=i, column=1, value=doc)
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        c.border = borde
        c.font = Font(name='Arial', size=9)
        for j, tipo in enumerate(TIPOS, 2):
            estilo(ws2.cell(row=i, column=j, value=int(row[tipo + '_orig'])), bg=COLORES[tipo])
        estilo(ws2.cell(row=i, column=6, value=int(row['TOTAL_orig'])), bold=True, bg=COLORES['total'])
        for j, tipo in enumerate(TIPOS, 7):
            estilo(ws2.cell(row=i, column=j, value=int(row[tipo + '_sub'])), bg=COLORES[tipo])
        estilo(ws2.cell(row=i, column=11, value=int(row['TOTAL_sub'])), bold=True, bg=COLORES['total'])

    # Hoja 3: Estadísticas de nota
    ws3 = wb.create_sheet('Notas')
    for col, w in zip('ABCDE', [14, 12, 12, 12, 12]):
        ws3.column_dimensions[col].width = w

    ws3.merge_cells('A1:E1')
    estilo(ws3['A1'], bold=True, bg=COLORES['titulo'], color='FFFFFF')
    ws3['A1'] = 'ESTADÍSTICAS DE NOTA EN LA SUBMUESTRA'
    ws3.row_dimensions[1].height = 24

    for col, cab in enumerate(['Tipo', 'Media', 'Mediana', 'Mín', 'Máx'], 1):
        estilo(ws3.cell(row=2, column=col, value=cab), bold=True, bg=COLORES['header'])

    for i, tipo in enumerate(['Directa', 'Caso', 'Compleja'], 3):
        sub = df_sub[df_sub['tipo'] == tipo].copy()
        sub['nota_num'] = pd.to_numeric(sub['nota'], errors='coerce')
        valores = [
            tipo,
            f"{sub['nota_num'].mean():.2f}" if not sub.empty else '-',
            f"{sub['nota_num'].median():.2f}" if not sub.empty else '-',
            f"{sub['nota_num'].min():.1f}" if not sub.empty else '-',
            f"{sub['nota_num'].max():.1f}" if not sub.empty else '-',
        ]
        for col, val in enumerate(valores, 1):
            estilo(ws3.cell(row=i, column=col, value=val), bg=COLORES.get(tipo, 'FFFFFF'))

    wb.save(ARCHIVO_SALIDA_EXCEL)
    print(f"\n  Excel comparativo: '{ARCHIVO_SALIDA_EXCEL}'")
    print("="*70)


if __name__ == '__main__':
    main()


 SUBMUESTREO ESTRATIFICADO POR NOTA
  Dataset original:    263 preguntas
  Documentos:          20
  Objetivo:            160 preguntas
  Seed:                42

  Candidatas totales (válidas + Negativas): 263
    Directa    71
    Caso       64
    Compleja   50
    Negativa   78

  Directa    →  40 seleccionadas
  Caso       →  40 seleccionadas
  Compleja   →  40 seleccionadas
  Negativa   →  40 seleccionadas

  JSON submuestreado: 'dataset_submuestreado.json' (160 preguntas)

 VERIFICACIÓN DEL SUBMUESTREO

  Distribución por tipo:
    Directa     40  (25.0%)
    Caso        40  (25.0%)
    Compleja    40  (25.0%)
    Negativa    40  (25.0%)

  Cobertura documental: 20/20 documentos
  [OK] Todos los documentos representados

  Estadísticas de nota en la submuestra (no-Negativas):
    Media:    23.46
    Mediana:  10.00
    Mín:      8.0
    Máx:      100.0

  Excel comparativo: 'analisis_submuestreo.xlsx'
